In [1]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
X = pd.read_csv('/Users/veron/Desktop/predicting_credit_card_payment/notebooks/credit_card_cleaned.csv')

In [3]:
feature_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
X.dropna(axis=0, subset=['default.payment.next.month'], inplace=True)
y = X['default.payment.next.month']
X = X[feature_cols]


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [5]:
logreg = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [7]:
model = XGBClassifier(eval_metric = 'logloss', random_state=0)

In [8]:
for name, m in [('logreg', logreg), ('xgb', model)]:
    auc = cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc')
    ap = cross_val_score(m, X_train, y_train, cv=cv, scoring='average_precision')
    print('%s: AUC %.4f +/- %.4f, PR-AUC %.4f' % (name, auc.mean(), auc.std(), ap.mean()))

logreg: AUC 0.7256 +/- 0.0074, PR-AUC 0.5043
xgb: AUC 0.7585 +/- 0.0038, PR-AUC 0.5245
